# AI-Based Object Counting — Grounding DINO

## Matrix Smart Technologies AI Engineer Assessment

**Model:** `IDEA-Research/grounding-dino-tiny`

**Detection prompt:** `screw. bolt.`

**Box threshold:** `0.20`

**Text threshold:** `0.20`

### Execution
1. Mount Google Drive.
2. Execute the dependency, dataset discovery, and model setup cell.
3. Execute the full-dataset inference cell.
4. Inspect the generated CSV files, overlays, and detection-region masks.

The model performs zero-shot object detection. The generated masks
represent rectangular detection regions, not pixel-level segmentation.


## 1. Dependencies, dataset discovery, and model initialization

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:

# Dependencies
!pip -q install transformers==4.51.3 accelerate

import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

# Project paths
PROJECT_ROOT = Path(
    "/content/gdrive/MyDrive/Matrix_AI_Assessment"
)

SUBMISSION_ROOT = PROJECT_ROOT / "Matrix_AI_Assessment_Submission"
AI_ROOT = SUBMISSION_ROOT / "AI"

DATASET_NAMES = [
    "ScrewAndBolt_20240713",
    "Screws_2024_07_15"
]

# Locate both datasets
dataset_paths = {}

for dataset_name in DATASET_NAMES:
    matches = [
        path for path in PROJECT_ROOT.rglob(dataset_name)
        if path.is_dir()
        and SUBMISSION_ROOT not in path.parents
    ]

    if matches:
        dataset_paths[dataset_name] = matches[0]

missing = set(DATASET_NAMES) - set(dataset_paths)

if missing:
    raise FileNotFoundError(
        f"Dataset folders not found: {sorted(missing)}"
    )

print("Datasets:")
for name, path in dataset_paths.items():
    print(name, "->", path)

# Model configuration
MODEL_ID = "IDEA-Research/grounding-dino-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Inference device:", device)
print("Loading Grounding DINO...")

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForZeroShotObjectDetection.from_pretrained(
    MODEL_ID
).to(device)

model.eval()

print("Model loaded successfully.")


## 2. Full dataset inference and artifact generation

In [ ]:

# ============================================================
# PHASE 5C: FULL DATASET GROUNDING DINO INFERENCE
# Selected thresholds: box=0.20, text=0.20
# ============================================================

from pathlib import Path
from PIL import Image, ImageDraw
import pandas as pd
import numpy as np
import torch

FULL_ROOT = AI_ROOT / "outputs" / "grounding_dino_full"
FULL_OVERLAYS = FULL_ROOT / "overlays"
FULL_MASKS = FULL_ROOT / "masks"

FULL_OVERLAYS.mkdir(parents=True, exist_ok=True)
FULL_MASKS.mkdir(parents=True, exist_ok=True)

BOX_THRESHOLD = 0.20
TEXT_THRESHOLD = 0.20
TEXT_PROMPT = "screw. bolt."

prediction_rows = []
object_rows = []

for dataset_name, dataset_path in dataset_paths.items():

    image_files = sorted(
        p for p in Path(dataset_path).rglob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
    )

    print(f"\nProcessing {dataset_name}: {len(image_files)} images")

    for image_path in image_files:

        image = Image.open(image_path).convert("RGB")

        inputs = processor(
            images=image,
            text=TEXT_PROMPT,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)

        detected = processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            box_threshold=BOX_THRESHOLD,
            text_threshold=TEXT_THRESHOLD,
            target_sizes=[(image.height, image.width)]
        )[0]

        boxes = detected["boxes"].cpu().numpy()
        scores = detected["scores"].cpu().numpy()
        labels = detected.get("text_labels", detected["labels"])

        count = len(boxes)

        prediction_rows.append({
            "dataset": dataset_name,
            "filename": image_path.name,
            "predicted_count": count,
            "status": "success"
        })

        annotated = image.copy()
        draw = ImageDraw.Draw(annotated)
        mask = Image.new("L", image.size, 0)
        mask_draw = ImageDraw.Draw(mask)

        for idx, (box, score, label) in enumerate(
            zip(boxes, scores, labels), start=1
        ):

            x1, y1, x2, y2 = map(float, box)

            object_rows.append({
                "dataset": dataset_name,
                "filename": image_path.name,
                "object_id": idx,
                "label": str(label),
                "confidence": float(score),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2
            })

            draw.rectangle(
                [x1, y1, x2, y2],
                outline="red",
                width=3
            )

            draw.text(
                (x1, max(0, y1 - 15)),
                f"{label} {score:.2f}",
                fill="red"
            )

            mask_draw.rectangle(
                [x1, y1, x2, y2],
                fill=255
            )

        # Dataset-specific filenames prevent collisions
        output_name = f"{dataset_name}_{image_path.name}"

        annotated.save(FULL_OVERLAYS / output_name)
        mask.save(FULL_MASKS / f"{image_path.stem}_{dataset_name}_mask.png")

        print(f"{image_path.name}: {count} detections")

# Save independent AI outputs
pred_df = pd.DataFrame(prediction_rows)
obj_df = pd.DataFrame(object_rows)

pred_df.to_csv(FULL_ROOT / "grounding_dino_predictions.csv", index=False)
obj_df.to_csv(FULL_ROOT / "grounding_dino_object_details.csv", index=False)

print("\n========== FULL DATASET INFERENCE COMPLETE ==========")
print("Images processed:", len(pred_df))
print("Total detections:", len(obj_df))
print("Prediction CSV:", FULL_ROOT / "grounding_dino_predictions.csv")
print("Object details:", FULL_ROOT / "grounding_dino_object_details.csv")
print("Overlays:", FULL_OVERLAYS)
print("Detection-region masks:", FULL_MASKS)

display(pred_df)


## Output artifacts

- `grounding_dino_predictions.csv`
- `grounding_dino_object_details.csv`
- Annotated detection overlays
- Rectangular detection-region masks

Outputs are written to `AI/outputs/grounding_dino_full`.
